# CEFR 3-Band Classification with a single XGBoost (multiclass)

One **XGBoost** model that predicts the band **directly as 3-class multiclass** - for each
learner it outputs P(band0), P(band1), P(band2) and the predicted band is the **argmax**
("which of the 3 bands is this candidate").

```
features -> XGBoost (3-class softprob) -> [P0, P1, P2] -> argmax = band
                                                       -> expected value = 0-100 score
```

**Bands:** `A1-A2` (0) < `B1` (1) < `B2-C1-C2` (2). Baseline to beat: 77%; target >=82%.

**What this notebook shows**
1. **Results** - train / test / full accuracy + confusion matrices (on the multiclass argmax).
2. **0-100 score** - expected value `P0*0 + P1*50 + P2*100` per learner.
3. **Feature importance (full)** - XGBoost native gain + permutation importance, to sections.
4. **Distribution in bins** - the 0-100 score histogram and per-band ranges.

## 0. Imports

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             cohen_kappa_score, confusion_matrix, classification_report)
from xgboost import XGBClassifier

print("xgboost loaded")

## 1. Load your data  <-- FILL THIS IN

In [ ]:
# TODO: assign your dataframe
df = None
# e.g. df = pd.read_csv("your_file.csv")

## 2. Columns  <-- FILL THIS IN

In [ ]:
FEATURE_COLS = []                 # one feature per group (8-11)

# OPTIONAL: map feature -> section, for section-level importance roll-up
FEATURE_GROUPS = {}               # e.g. {"lex_div": "Vocabulary", "mlu": "Grammar"}

ID_COL, LOCATION_COL, SPLIT_COL, LABEL_COL = "ciid", "location", "split", "cefr"
TRAIN_VALUE, TEST_VALUE = "train", "test"
META_COLS = [ID_COL, LOCATION_COL, SPLIT_COL, LABEL_COL]

## 3. Configuration

In [ ]:
RANDOM_STATE = 42
BAND_MAP = {"A1": 0, "A2": 0, "B1": 1, "B2": 2, "C1": 2, "C2": 2}
BAND_NAMES = ["A1-A2", "B1", "B2-C1-C2"]
N_BANDS = 3

BAND_ANCHORS = np.array([0.0, 50.0, 100.0])   # 0-100 score = sum_k P(band_k) * anchor_k
PERM_REPEATS = 20
BASELINE_ACC, TARGET_ACC = 0.77, 0.82

## 4. Build train / test  (from the `split` column)

In [ ]:
assert df is not None and len(FEATURE_COLS) > 0, "Fill in df and FEATURE_COLS."
miss = [c for c in FEATURE_COLS + META_COLS if c not in df.columns]
assert not miss, f"missing columns: {miss}"

def to_band(s):
    s = pd.Series(s)
    if s.dtype.kind in "iuf" and set(pd.unique(s.dropna())) <= {0, 1, 2}:
        return s.astype(int).to_numpy()
    key = s.astype(str).str.strip().str.upper().str.replace(" ", "", regex=False)
    m = key.map(BAND_MAP); assert m.notna().all(), f"unmapped: {key[m.isna()].unique()}"
    return m.astype(int).to_numpy()

sp = df[SPLIT_COL].astype(str).str.strip().str.lower()
train_df, test_df = df.loc[sp == TRAIN_VALUE].copy(), df.loc[sp == TEST_VALUE].copy()
X_train, X_test = train_df[FEATURE_COLS].astype(float), test_df[FEATURE_COLS].astype(float)
y_train, y_test = to_band(train_df[LABEL_COL]), to_band(test_df[LABEL_COL])

print(f"train/test rows: {len(X_train)}/{len(X_test)} | dropped bad flags: {(~sp.isin([TRAIN_VALUE, TEST_VALUE])).sum()}")
print("train band counts:", dict(zip(*np.unique(y_train, return_counts=True))))
print("test  band counts:", dict(zip(*np.unique(y_test,  return_counts=True))))

## 5. Utilities  (0-100 score)

In [ ]:
def proba_to_score(proba, anchors=BAND_ANCHORS):
    """0-100 score = expected value of the band index: P0*0 + P1*50 + P2*100."""
    return np.asarray(proba) @ np.asarray(anchors, float)

def band_metrics(y, pred):
    return dict(acc=accuracy_score(y, pred), bal=balanced_accuracy_score(y, pred),
                mf1=f1_score(y, pred, average="macro"),
                qwk=cohen_kappa_score(y, pred, weights="quadratic"))
print("utilities ready")

## 6. Train the single XGBoost and get results

Direct 3-class multiclass (`multi:softprob`), shallow trees + regularisation for n~220.
The predicted band is the **argmax** of the three probabilities.

In [ ]:
xgb = XGBClassifier(
    objective="multi:softprob", num_class=N_BANDS, eval_metric="mlogloss",
    n_estimators=300, learning_rate=0.05, max_depth=3, min_child_weight=3,
    subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0, gamma=0.0,
    tree_method="hist", random_state=RANDOM_STATE, n_jobs=-1, verbosity=0,
    importance_type="gain",
)
pipe = Pipeline([("impute", SimpleImputer(strategy="median")), ("model", xgb)])
pipe.fit(X_train, y_train)

p_tr, p_te = pipe.predict_proba(X_train), pipe.predict_proba(X_test)
pred_tr, pred_te = p_tr.argmax(1), p_te.argmax(1)            # multiclass argmax = band
s_tr, s_te = proba_to_score(p_tr), proba_to_score(p_te)      # 0-100 score

y_full = np.concatenate([y_train, y_test]); pred_full = np.concatenate([pred_tr, pred_te])
mt = band_metrics(y_test, pred_te)
flag = "PASS >=82%" if mt["acc"] >= TARGET_ACC else ("beats 77%" if mt["acc"] >= BASELINE_ACC else "below 77%")
print("=== XGBoost band accuracy (multiclass argmax) ===")
print(f"  TRAIN {accuracy_score(y_train, pred_tr):.3f} | TEST {mt['acc']:.3f}   <-- honest  [{flag}]")
print(f"  FULL (train+test) {accuracy_score(y_full, pred_full):.3f}   (inflated - do not quote)")
print(f"  test: balanced {mt['bal']:.3f} | macroF1 {mt['mf1']:.3f} | QWK {mt['qwk']:.3f}")
print(f"  (baseline {BASELINE_ACC:.0%}, target {TARGET_ACC:.0%})")

### Confusion matrices - train / test / full

In [ ]:
for title, yt, yp in [("TRAIN", y_train, pred_tr), ("TEST", y_test, pred_te),
                      ("FULL (train+test)", y_full, pred_full)]:
    print(f"\n----- {title}  (accuracy {accuracy_score(yt, yp):.3f}) -----")
    display(pd.DataFrame(confusion_matrix(yt, yp, labels=[0, 1, 2]),
                         index=[f"true {b}" for b in BAND_NAMES],
                         columns=[f"pred {b}" for b in BAND_NAMES]))
print("\n----- TEST classification report -----")
print(classification_report(y_test, pred_te, target_names=BAND_NAMES))

## 7. Feature importance (full dataset)

- **Native gain:** total loss improvement from all splits on each feature (XGBoost's own).
- **Permutation on FULL:** shuffle a feature, re-run the model, measure the drop in band
  accuracy on the whole dataset. Both normalised to a share-of-total percentage and rolled up
  to sections.

In [ ]:
X_full = pd.concat([X_train, X_test])
model = pipe.named_steps["model"]

# native gain importance (aligned to FEATURE_COLS order)
gain = np.asarray(model.feature_importances_, float)

# permutation importance on the FULL dataset (drop in argmax band accuracy)
def acc_of(model_pipe, X, y):
    return accuracy_score(y, model_pipe.predict_proba(X).argmax(1))

rng = np.random.default_rng(RANDOM_STATE)
base = acc_of(pipe, X_full, y_full)
perm = []
for col in FEATURE_COLS:
    drops = []
    for _ in range(PERM_REPEATS):
        Xp = X_full.copy(); Xp[col] = rng.permutation(Xp[col].to_numpy())
        drops.append(base - acc_of(pipe, Xp, y_full))
    perm.append(float(np.mean(drops)))

imp = pd.DataFrame({"feature": FEATURE_COLS, "gain": gain, "perm_full": perm})
imp["section"] = imp["feature"].map(lambda f: FEATURE_GROUPS.get(f, f))
def pct(col):
    p = imp[col].clip(lower=0)
    return (100 * p / p.sum()).round(1) if p.sum() > 0 else p * 0.0
imp["gain_pct"] = pct("gain")
imp["perm_pct"] = pct("perm_full")
imp = imp.sort_values("perm_full", ascending=False).reset_index(drop=True)

print(f"full-data band accuracy (permutation baseline): {base:.3f}")
print("gain = XGBoost native | perm_full = drop in band accuracy when shuffled | *_pct sums to ~100%\n")
display(imp[["feature", "section", "gain", "gain_pct", "perm_full", "perm_pct"]].round(4))

drop = imp[imp["perm_full"] <= 0]["feature"].tolist()
if drop:
    print("no help (permutation <= 0):", drop)

if FEATURE_GROUPS:
    print("\nSECTION-level percentages (each column sums to ~100%):")
    display(imp.groupby("section")[["gain_pct", "perm_pct"]].sum()
               .sort_values("perm_pct", ascending=False).round(1))

try:
    import matplotlib.pyplot as plt
    top = imp.head(12).iloc[::-1]
    plt.figure(figsize=(7, 4))
    plt.barh(top["feature"], top["perm_pct"], color="#3b6ea5")
    plt.title("XGBoost permutation importance (full, %)"); plt.xlabel("% of total"); plt.tight_layout(); plt.show()
except Exception as e:
    print("(plot skipped:", e, ")")

## 8. The 0-100 score and its distribution in bins

`score = P0*0 + P1*50 + P2*100` (expected value). The histogram (train / test / full) and the
per-band score ranges. Raw expected-value scores tend to be U-shaped; see
`cefr_2_methods_reshaped.ipynb` for bell reshaping.

In [ ]:
s_full = proba_to_score(np.vstack([p_tr, p_te]))
print("0-100 score by band (full data):")
for b in range(N_BANDS):
    v = s_full[pred_full == b]
    if len(v):
        print(f"  {BAND_NAMES[b]:<9} {v.min():.0f}-{v.max():.0f}  median {np.median(v):.0f}  n={len(v)}")
print("\nscore deciles (0-10,...,90-100):", list(np.histogram(s_full, bins=10, range=(0, 100))[0]))

try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.2), sharey=True)
    for a, (title, sc) in zip(ax, [("train", s_tr), ("test", s_te), ("full", s_full)]):
        a.hist(sc, bins=20, range=(0, 100), color="#3b6ea5")
        a.set_title(f"XGB 0-100 score - {title}"); a.set_xlim(0, 100); a.set_xlabel("0-100 score")
    ax[0].set_ylabel("learners"); plt.tight_layout(); plt.show()
except Exception as e:
    print("(plot skipped:", e, ")")

## 9. Final predictions - full dataset

Per learner: the three band probabilities, the **0-100 score**, the predicted band, with
`ciid` / `split` / `region`.

In [ ]:
n_tr = len(X_train)
out = pd.DataFrame({
    ID_COL:   np.concatenate([train_df[ID_COL].values, test_df[ID_COL].values]),
    "region": np.concatenate([train_df[LOCATION_COL].values, test_df[LOCATION_COL].values]),
    "split":  ["train"] * n_tr + ["test"] * len(X_test),
    "true_band": [BAND_NAMES[i] for i in y_full],
    "P0": np.round(np.concatenate([p_tr[:, 0], p_te[:, 0]]), 4),
    "P1": np.round(np.concatenate([p_tr[:, 1], p_te[:, 1]]), 4),
    "P2": np.round(np.concatenate([p_tr[:, 2], p_te[:, 2]]), 4),
    "score": np.round(s_full, 2),
    "pred_band": [BAND_NAMES[i] for i in pred_full],
})
with pd.option_context("display.max_rows", 400, "display.max_columns", 60):
    display(out)
# out.to_csv("xgb_predictions.csv", index=False)

## Notes

- **One model, plain multiclass:** the band is the argmax of the 3-class softprob output - no
  Frank-Hall, no hierarchy. The **0-100 score** is the expected value of the band index.
- **Importance to quote:** permutation `perm_pct` (share of real band accuracy). Native `gain`
  is XGBoost's internal view.
- **Small n:** shallow trees + `reg_lambda` + subsampling guard against overfitting; treat exact
  importance numbers as approximate (ordering is the signal).
- **Knobs:** `max_depth`, `n_estimators`, `learning_rate`, `reg_lambda`. Add light CV tuning to
  push accuracy.